# IRWA Final Project — Part 2: Indexing and Evaluation

Corpus: `data/fashion_products_dataset.json`. Same pre-processing as Part 1 (`preprocess`, `details_to_text`, `normalize_sub_category`).

In [4]:
import json
import math
import re
import string
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

for res in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(res, quiet=True)

ModuleNotFoundError: No module named 'pandas'

## 0. Pre-processing (from Part 1)
The same functions as in Part 1, so documents and queries are processed identically.

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
PUNCT_TABLE = str.maketrans({c: " " for c in string.punctuation})


def preprocess(text, stem=True):
    """Clean, tokenize, remove punctuation/stop words/digit tokens and (optionally) stem a string."""
    if not isinstance(text, str):
        return []
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    text = re.sub(r"'s\b", "", text)
    text = re.sub(r"\bt[\s-]?shirts?\b", "tshirt", text)
    text = text.translate(PUNCT_TABLE)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1 and not any(c.isdigit() for c in t)]
    if stem:
        tokens = [STEMMER.stem(t) for t in tokens]
    return tokens


with open("../../data/fashion_products_dataset.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

DROP_DETAIL_KEYS = {"Style Code", "Fabric Care", "Sales Package", "Country of Origin", "Pack of", "Reversible",
                    "Size", "Other Details", "Model Name", "Suitable For", "Brand Color", "Brand Fit", "Sleeve Type"}
MULTIWORD_CATEGORIES = [c for c in df["category"].unique() if " " in c]


def details_to_text(details):
    """Join the values of the informative product_details keys into one string."""
    return " ".join(str(v) for d in details for k, v in d.items() if k not in DROP_DETAIL_KEYS)


def normalize_sub_category(value):
    """Map values like 'Roy Clothing and Accessories' to 'Clothing and Accessories'."""
    for cat in MULTIWORD_CATEGORIES:
        if value.endswith(cat) and value != cat:
            return cat
    return value


df["sub_category"] = df["sub_category"].map(normalize_sub_category)
df["tokens"] = (df["title"].map(preprocess) + df["description"].map(preprocess)
                + df["product_details"].map(lambda d: preprocess(details_to_text(d)))
                + df["brand"].map(preprocess) + df["sub_category"].map(preprocess))
corpus = df.set_index("pid", drop=False)
print("Documents:", len(corpus))

Documents: 28080


---
## PART 1: Indexing

### 1.1 Inverted index
`index[term]` → list of the `pid`s of the documents that contain the term (vocabulary structure seen in the labs).
We also keep, for each document, its term frequencies (`tf`), needed by TF-IDF.

In [ ]:
def build_index(docs):
    """docs: iterable of (pid, tokens). Returns the inverted index and the term frequencies per document."""
    index = defaultdict(list)   # term -> [pid, ...]
    tf = {}                     # pid  -> Counter({term: count})
    for pid, tokens in docs:
        counts = Counter(tokens)
        tf[pid] = counts
        for term in counts:
            index[term].append(pid)
    return index, tf


index, tf = build_index(zip(df["pid"], df["tokens"]))
N = len(tf)
print("Vocabulary size:", len(index))
print("Example — 'sweatshirt':", index["sweatshirt"][:5], "...", len(index["sweatshirt"]), "documents")

Vocabulary size: 5058
Example — 'sweatshirt': ['SWSFYD33P2XCBWFG', 'SWSFYD3KMGTBZEJZ', 'SWSFYD38Y2MAT9FQ', 'SWSFVEV2YFEJKUCK', 'SWSFVEV2BWZJDJ46'] ... 1417 documents


**Conjunctive (AND) queries:** a document is returned only if it contains *all* the query terms → intersection of the posting lists.
We start from the shortest list, so the intersection is cheaper.

In [ ]:
def conjunctive_search(query):
    """Return the set of pids that contain every term of the query."""
    terms = set(preprocess(query))
    if not terms:
        return set()
    postings = sorted((set(index.get(t, [])) for t in terms), key=len)
    result = postings[0]
    for p in postings[1:]:
        result &= p
    return result


for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue"]:
    print(f"{q!r}: {len(conjunctive_search(q))} documents")

'women full sleeve sweatshirt cotton': 467 documents
'men slim jeans blue': 216 documents


### 1.2 Test queries
We look at the terms with the highest document frequency and combine popular terms (gender, product type, fit, colour, use) with
less frequent but more specific ones, so that each query is realistic and returns a manageable number of results.

In [ ]:
doc_freq = pd.Series({t: len(p) for t, p in index.items()}).sort_values(ascending=False)
print(doc_freq.head(40).to_dict())

{'cotton': 19610, 'sleev': 17909, 'topwear': 15310, 'neck': 14578, 'men': 14141, 'women': 13699, 'blend': 13593, 'regular': 13089, 'tshirt': 12902, 'solid': 11989, 'round': 9497, 'print': 9484, 'casual': 9480, 'half': 8062, 'fit': 7777, 'wear': 7717, 'full': 7156, 'blue': 6978, 'comfort': 6845, 'pack': 5924, 'black': 5898, 'pure': 5498, 'slim': 5422, 'fabric': 5059, 'look': 5050, 'design': 4948, 'short': 4941, 'shirt': 4809, 'multicolor': 4752, 'made': 4471, 'cloth': 3893, 'bottomwear': 3725, 'white': 3606, 'collar': 3393, 'color': 3389, 'day': 3301, 'pair': 3292, 'button': 3139, 'polo': 3029, 'qualiti': 2964}


In [ ]:
test_queries = {
    3: "women hooded jacket",
    4: "men running shoes",
    5: "women yellow kurta",
    6: "men regular fit formal shirt",
    7: "men sports shorts",
}
pd.DataFrame([{"query_id": qid, "query": q, "terms": preprocess(q), "AND results": len(conjunctive_search(q))}
              for qid, q in test_queries.items()])

,query_id,query,terms,AND results
0,3,women hooded jacket,"[women, hood, jacket]",80
1,4,men running shoes,"[men, run, shoe]",76
2,5,women yellow kurta,"[women, yellow, kurta]",39
3,6,men regular fit formal shirt,"[men, regular, fit, formal, shirt]",218
4,7,men sports shorts,"[men, sport, short]",196


- **women hooded jacket**: winter wear with an attribute (*hooded*) that appears in titles and product_details.
- **men running shoes**: footwear, a much smaller category than topwear.
- **women yellow kurta**: colour + an Indian garment, very specific.
- **men regular fit formal shirt**: long query with very frequent terms (*regular*, *fit*, *shirt*) → tests IDF.
- **men sports shorts**: *sports* also appears in descriptions of many other items → tests ranking quality.

### 1.3 TF-IDF ranking
- Document weight of term *t*: `w(t,d) = (1 + log tf(t,d)) · idf(t)`, with `idf(t) = log(N / df(t))`
- Query weight: the same formula with the term counts of the query
- Score: cosine similarity between the query and document vectors (document vectors are normalised by their length).

In [ ]:
idf = {t: math.log(N / len(p)) for t, p in index.items()}

doc_norm = {pid: math.sqrt(sum(((1 + math.log(c)) * idf[t]) ** 2 for t, c in counts.items())) or 1.0
            for pid, counts in tf.items()}


def tfidf_score(query_terms, pid):
    """Cosine similarity between the TF-IDF vectors of the query and of document pid."""
    q_counts = Counter(t for t in query_terms if t in idf)
    q_norm = math.sqrt(sum(((1 + math.log(c)) * idf[t]) ** 2 for t, c in q_counts.items())) or 1.0
    dot = sum((1 + math.log(qc)) * idf[t] * (1 + math.log(tf[pid][t])) * idf[t]
              for t, qc in q_counts.items() if t in tf[pid])
    return dot / (q_norm * doc_norm[pid])


def search(query, top_k=None):
    """AND retrieval + TF-IDF ranking. Returns a DataFrame with rank, pid, score and title."""
    terms = preprocess(query)
    ranked = sorted(((pid, tfidf_score(terms, pid)) for pid in conjunctive_search(query)),
                    key=lambda x: (-x[1], x[0]))[:top_k]
    res = pd.DataFrame(ranked, columns=["pid", "score"])
    res.insert(0, "rank", range(1, len(res) + 1))
    res["score"] = res["score"].round(4)
    res["title"] = res["pid"].map(corpus["title"])
    return res


for qid, q in test_queries.items():
    print(f"\nQuery {qid}: {q}")
    print(search(q, top_k=5).to_string(index=False))


Query 3: women hooded jacket
 rank              pid  score                                 title
    1 JCKFVEVFZ6BGWHFZ 0.6129 Full Sleeve Solid Women Sports Jacket
    2 JCKFUFGZZSPGSYSS 0.5916 Full Sleeve Solid Women Casual Jacket
    3 JCKFRAJTUACHZVHP 0.5751 Full Sleeve Solid Women Casual Jacket
    4 JCKFPRBPRGG6BCSA 0.5710 Full Sleeve Solid Women Casual Jacket
    5 JCKFRAJTVVHJJGGW 0.5690 Full Sleeve Solid Women Casual Jacket

Query 4: men running shoes
 rank              pid  score                                       title
    1 SHOEP5GUQYYFKYSN 0.6615            Running Shoes For Women  (White)
    2 SHOESS9ZEZ6TEQUJ 0.6493         Running Shoes For Men  (Multicolor)
    3 SHOESS9ZGMH7CURM 0.6493         Running Shoes For Men  (Multicolor)
    4 SHOEKNRJGEYJCDQK 0.5076  SPRINT Running Shoes For Men  (Blue, Grey)
    5 SHOEKNRJ34UCMAPA 0.5059 SPRINT Running Shoes For Men  (Black, Grey)

Query 5: women yellow kurta
 rank              pid  score                                

---
## PART 2: Evaluation

### 2.1 Metrics
Every function receives `rels`: the relevance labels (1/0) of the retrieved documents **in ranking order**,
and, when needed, `n_relevant`: the total number of relevant documents for the query.

In [ ]:
def precision_at_k(rels, k):
    return sum(rels[:k]) / k


def recall_at_k(rels, k, n_relevant):
    return sum(rels[:k]) / n_relevant if n_relevant else 0.0


def avg_precision_at_k(rels, k, n_relevant):
    """Mean of P@i over the positions i <= k that hold a relevant document, divided by min(n_relevant, k)."""
    hits, total = 0, 0.0
    for i, r in enumerate(rels[:k], start=1):
        if r:
            hits += 1
            total += hits / i
    denom = min(n_relevant, k)
    return total / denom if denom else 0.0


def f1_at_k(rels, k, n_relevant):
    p, r = precision_at_k(rels, k), recall_at_k(rels, k, n_relevant)
    return 2 * p * r / (p + r) if p + r else 0.0


def mean_average_precision(rels_per_query, k, n_relevant_per_query):
    return float(np.mean([avg_precision_at_k(r, k, n) for r, n in zip(rels_per_query, n_relevant_per_query)]))


def reciprocal_rank(rels, k):
    for i, r in enumerate(rels[:k], start=1):
        if r:
            return 1 / i
    return 0.0


def mean_reciprocal_rank(rels_per_query, k):
    return float(np.mean([reciprocal_rank(r, k) for r in rels_per_query]))


def dcg_at_k(rels, k):
    return sum(r / math.log2(i + 1) for i, r in enumerate(rels[:k], start=1))


def ndcg_at_k(rels, k, n_relevant):
    """DCG@k divided by the DCG@k of the ideal ranking (all n_relevant relevant documents first)."""
    ideal = dcg_at_k([1] * n_relevant, k)
    return dcg_at_k(rels, k) / ideal if ideal else 0.0


toy = [1, 0, 1, 0, 0, 1]  # sanity check, 3 relevant documents in total
print(precision_at_k(toy, 3), recall_at_k(toy, 3, 3), avg_precision_at_k(toy, 6, 3), f1_at_k(toy, 3, 3),
      reciprocal_rank(toy, 6), round(ndcg_at_k(toy, 6, 3), 3))

0.6666666666666666 0.6666666666666666 0.7222222222222222 0.6666666666666666 1.0 0.871


In [ ]:
def evaluate(rels_per_query, n_relevant_per_query, names, k=10):
    """Table with all metrics per query (+ MAP and MRR), rounded to 3 decimals."""
    rows = []
    for name, rels, n in zip(names, rels_per_query, n_relevant_per_query):
        rows.append({"query": name,
                     f"P@{k}": precision_at_k(rels, k), f"R@{k}": recall_at_k(rels, k, n),
                     f"AP@{k}": avg_precision_at_k(rels, k, n), f"F1@{k}": f1_at_k(rels, k, n),
                     f"RR@{k}": reciprocal_rank(rels, k), f"NDCG@{k}": ndcg_at_k(rels, k, n)})
    table = pd.DataFrame(rows).set_index("query").round(3)
    summary = pd.Series({f"MAP@{k}": mean_average_precision(rels_per_query, k, n_relevant_per_query),
                         f"MRR@{k}": mean_reciprocal_rank(rels_per_query, k)}).round(3)
    return table, summary

### 2.2 Validation queries (`validation_labels.csv`)
Relevance is only known for the documents listed in the file, so we run our search engine and keep, in ranking order, the retrieved documents that have a label.
Labelled documents that our AND search does not return count as relevant documents not retrieved (they lower recall). K = 10.

In [ ]:
labels = pd.read_csv("../../data/validation_labels.csv")
validation_queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
K = 10

val_rels, val_n_rel = [], []
for qid, q in validation_queries.items():
    judged = labels[labels["query_id"] == qid].set_index("pid")["labels"]
    ranking = search(q)
    ranking = ranking[ranking["pid"].isin(judged.index)]
    val_rels.append(ranking["pid"].map(judged).astype(int).tolist())
    val_n_rel.append(int(judged.sum()))

val_table, val_summary = evaluate(val_rels, val_n_rel, list(validation_queries.values()), k=K)
val_table

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query,,,,,,
women full sleeve sweatshirt cotton,1.0,0.769,1.000,0.87,1.0,1.000
men slim jeans blue,0.6,0.600,0.569,0.60,1.0,0.716


In [ ]:
val_summary

MAP@10    0.785
MRR@10    1.000
dtype: float64

### 2.3 Our own queries — we act as expert judges

**a.** For each of our 5 test queries we judged the **top 20** documents returned by the system, reading title, description and product_details,
with a binary label (1 = relevant, 0 = not relevant). The judgments are stored in `my_labels.csv`.

Criteria: the product must be the requested **type** of item, for the requested **gender**, and satisfy the requested attribute (hooded, yellow, formal, sports/running).

As the whole collection cannot be judged, the number of relevant documents per query is the number of relevant documents found among the judged ones
(so recall is measured relative to the judged pool).

In [ ]:
# Run once to create the file to judge (it is already filled in my_labels.csv)
to_judge = []
for qid, q in test_queries.items():
    res = search(q, top_k=20)
    res.insert(0, "query", q)
    res.insert(0, "query_id", qid)
    res["details"] = res["pid"].map(corpus["product_details"].map(details_to_text))
    to_judge.append(res)
pd.concat(to_judge).to_csv("to_judge.csv", index=False)

In [ ]:
my_labels = pd.read_csv("my_labels.csv").sort_values(["query_id", "rank"])

my_rels = [g["label"].astype(int).tolist() for _, g in my_labels.groupby("query_id")]
my_n_rel = [sum(r) for r in my_rels]
my_table, my_summary = evaluate(my_rels, my_n_rel, list(test_queries.values()), k=K)
my_table

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query,,,,,,
women hooded jacket,0.7,0.500,0.700,0.583,1.0,0.801
men running shoes,0.8,0.727,0.598,0.762,0.5,0.710
women yellow kurta,0.6,0.462,0.600,0.522,1.0,0.727
men regular fit formal shirt,1.0,0.500,1.000,0.667,1.0,1.000
men sports shorts,1.0,0.500,1.000,0.667,1.0,1.000


In [ ]:
my_summary

MAP@10    0.78
MRR@10    0.90
dtype: float64

**b. Discussion of the metrics**

| Metric | What it measures | What it ignores |
|---|---|---|
| **P@K** | Fraction of the first K results that are relevant (quality of the first page) | Order inside the top K; relevant documents not retrieved |
| **R@K** | Fraction of all relevant documents that appear in the top K (coverage) | Non-relevant results; order. It can only grow with K, while P@K usually decreases |
| **AP@K** | Average of the precision at each position that holds a relevant document | — it is order-aware: relevant documents at the top are rewarded |
| **F1@K** | Harmonic mean of P@K and R@K: high only if both are high | Order |
| **MAP** | Mean of AP over all queries: one number for the ranking quality of the whole system | Differences between queries (an easy query compensates a hard one) |
| **MRR** | Position of the *first* relevant result (1, 1/2, 1/3…): a user who needs only one good product | Everything after the first relevant result |
| **NDCG@K** | Relevant results discounted by log2 of their position, normalised by the ideal ranking (1 = perfect order) | — it supports graded relevance; with binary labels it is a position-aware precision |

**Analysis of our results (K = 10)**

- **Validation query 1** (*women full sleeve sweatshirt cotton*) is solved very well: P@10 = AP@10 = NDCG@10 = 1, all of the first 10 judged results are relevant. R@10 = 0.769 because there are 13 relevant documents and only 10 fit in the top 10.
- **Validation query 2** (*men slim jeans blue*) is much worse (P@10 = 0.6, AP@10 = 0.569). Only 8 of the 20 labelled documents are retrieved: the AND condition excludes every women's product, but the ground truth labels some *Women* jeans as relevant, and the TF-IDF ranking places 2 non-relevant men's jeans among the retrieved ones. MRR is still 1 because the first result is relevant, which shows how MRR hides problems further down the ranking.
- **Our queries**: *men regular fit formal shirt* and *men sports shorts* are easy (P@10 = AP@10 = NDCG@10 = 1): the query terms are exactly the words used in the titles.
- *men running shoes* has the highest P@10 among the imperfect queries (0.8) but the lowest RR (0.5) and AP (0.598): the first result is a women's shoe. P@10 does not see this, while AP, RR and NDCG penalise it. The cause is that the product has `sub_category = "Men's Footwear"`, so the term *men* is found in the combined field.
- *women hooded jacket* (P@10 = 0.7) retrieves hooded **sweatshirts** at positions 8–11: their description is literally *"Full Sleeve Hooded Jacket"*, so they contain all the terms, although the title says they are a different product type.
- *women yellow kurta* (P@10 = 0.6) mixes single kurtas with *kurta and pyjama sets*, which contain the same terms. The relevant kurtas at positions 12 and 15–20 show the effect of ranking: R@10 = 0.462.
- R@10 is ≤ 0.5 for most of our queries simply because each has 11–20 relevant documents in the judged pool of 20, and at most 10 can be in the top 10. This is why P and R must be read together (F1).
- Overall, MAP@10 is similar on both sets (0.785 validation, 0.780 ours), and MRR is high (1.0 and 0.9): the system usually puts a relevant product first, but the quality of the rest of the ranking varies a lot between queries.

**c. Limitations of the current system and proposed solutions**

| Problem | Observed effect | Possible solution |
|---|---|---|
| Strict conjunctive (AND) retrieval | A query term missing in a relevant product removes it (e.g. many labelled *men slim jeans blue* documents are not retrieved) | Disjunctive (OR) retrieval with ranking, or AND with OR fallback when there are few results |
| All fields merged with the same weight | A term in a long description weighs as much as one in the title | Field weighting (title > product_details > description), e.g. BM25F |
| Gender handled as a normal term | A women's shoe is the 1st result for *men running shoes* because its sub_category is *Men's Footwear*; women's jeans labelled relevant for query 2 can never be retrieved | Extract a gender field (from *Ideal For* and the title) and use it as a filter/boost; clean inconsistent sub_category values |
| TF-IDF ranking | No term-frequency saturation; many identical titles give ties | BM25; break ties with rating, discount, stock |
| No synonyms / spelling variants | *hoodie* vs *hooded*, *trainers* vs *shoes*, typos give 0 results | Query expansion with synonyms, spelling correction, embeddings |
| No product-type awareness / no phrases | Hooded sweatshirts for *hooded jacket*, kurta+pyjama sets for *kurta*; *full sleeve* matched as two independent words | Give more weight to the product type (last noun of the title / *Generic Name*), bigram or phrase index, proximity scoring |
| Numeric fields unused | Out-of-stock products are ranked like the rest; no price filters | Filters (price, rating, stock) and features in the ranking |
| Small and subjective ground truth | Only 20 judged documents per query, a single judge per document | More queries, deeper pools, several judges, graded labels (0/1/2) for NDCG |